# 01 — Data Acquisition

Downloads daily maximum near-surface air temperature (Tmax) from the
**Zhang et al. (2022) Global Sub-daily Heat Temperature Dataset (GSHTD)**
via Google Earth Engine and writes one NetCDF per city to `data/`.

**Run this notebook once to populate the cache.**  
Subsequent notebooks (`02_compute_baselines`, `03_threshold_analysis`) read
only from disk and never contact GEE.

Expected runtime: 30–90 minutes per city depending on GEE queue.

## Data source

| Property | Value |
|---|---|
| Citation | Zhang et al. (2022) |
| GEE collection | `projects/sat-io/open-datasets/global-daily-air-temp/latin_america` |
| Spatial resolution | 1 km |
| Temporal coverage | 2003-01-01 to 2020-12-31 |
| Variable pulled | `tmax` (band `b1`, raw value / 10 = °C) |

All three cities are in Latin America, so one collection covers the full scope.

In [2]:
import ee
import geemap
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils.heat import CITIES, extract_city_tmax, load_city

## GEE initialisation

Authenticate once with `earthengine authenticate` in a terminal if this fails.

In [2]:
ee.Initialize(project='tl-cities')
print('GEE initialised')

GEE initialised


## Configuration

Edit `DATA_DIR`, `START`, or `END` here if needed.  
If you change the date range, delete the existing `.nc` files and re-run.

In [4]:
DATA_DIR   = 'data'
START      = '2003-01-01'
END        = '2020-12-31'
RESOLUTION = 1000  # metres

print(f'Period     : {START} to {END}')
print(f'Resolution : {RESOLUTION} m')
print(f'Output dir : {DATA_DIR}/')
print()
for name, cfg in CITIES.items():
    print(f'  {name:10s}  bbox={cfg["bbox"]}  IBGE={cfg["ibge_code"]}')

Period     : 2003-01-01 to 2020-12-31
Resolution : 1000 m
Output dir : data/

  salvador    bbox={'west': -38.8, 'east': -38.2, 'south': -13.2, 'north': -12.7}  IBGE=2927408
  teresina    bbox={'west': -43.2, 'east': -42.5, 'south': -5.4, 'north': -4.8}  IBGE=2211001
  caceres     bbox={'west': -58.1, 'east': -57.3, 'south': -16.5, 'north': -15.7}  IBGE=5102504


## Extract temperature data

Each cell below pulls one city.  They are intentionally separate so you can
re-run a single city without touching the others.

`extract_city_tmax` skips the download if a cache file already exists.  
Pass `force_refresh=True` to overwrite.

### Salvador, Brazil

Coastal city, humid tropical climate. Bounding box covers the Salvador
metropolitan municipality (~738 km²).

In [4]:
extract_city_tmax('salvador', DATA_DIR, START, END, resolution=RESOLUTION)

Cache exists: data/salvador_tmax.nc  (pass force_refresh=True to re-download)


### Teresina, Brazil

Inland semi-arid city, one of the hottest capitals in Brazil.
Bounding box covers the urban core of Teresina municipality.

In [5]:
extract_city_tmax('teresina', DATA_DIR, START, END, resolution=RESOLUTION)

Cache exists: data/teresina_tmax.nc  (pass force_refresh=True to re-download)


### Cáceres, Brazil

Inland city in Mato Grosso at the edge of the Pantanal wetland.
Verify the bounding box in `CITIES['caceres']` against the GEE map on first
run — adjust `utils/heat.py` if the pixel footprint is wrong.

In [6]:
extract_city_tmax('caceres', DATA_DIR, START, END, resolution=RESOLUTION)

Cache exists: data/caceres_tmax.nc  (pass force_refresh=True to re-download)


## Verification

Quick sanity check: confirm shape, date range, and temperature bounds for
each city.  Expected Tmax range for Brazilian cities: roughly 20–45 °C.

In [7]:
for city in ['salvador', 'teresina', 'caceres']:
    da = load_city(city, DATA_DIR)
    tmin_val = float(da.min())
    tmax_val = float(da.max())
    n_nan    = int(da.isnull().sum())
    print(f'  Tmax range : {tmin_val:.1f} to {tmax_val:.1f} C')
    print(f'  NaN pixels : {n_nan:,}')
    print()

salvador: shape {'time': 6497, 'lat': 55, 'lon': 67}  2003-01-01 to 2020-12-30
  Tmax range : -14.2 to 88.6 C
  NaN pixels : 16,279,289

teresina: shape {'time': 6497, 'lat': 67, 'lon': 78}  2003-01-01 to 2020-12-30
  Tmax range : 4.5 to 67.1 C
  NaN pixels : 28,515

caceres: shape {'time': 6497, 'lat': 89, 'lon': 89}  2003-01-01 to 2020-12-30
  Tmax range : 13.7 to 53.2 C
  NaN pixels : 0



## Population data

Downloads annual WorldPop 100m total population (`WorldPop/GP/100m/pop`) for
each city and saves `data/{city}_population.nc` — shape `(year, lat, lon)`.

**Run once after acquiring temperature data.** Years 2003–2020 are requested;
any year not yet in WorldPop is proxied with the nearest available year.

Expected runtime: 5–20 minutes per city (18 × 1 GEE request per year).

In [8]:
from utils.heat import extract_city_population

POP_START = 2003
POP_END   = 2020
POP_RES   = 100  # metres


### Salvador, Brazil

In [9]:
extract_city_population('salvador', DATA_DIR, POP_START, POP_END, resolution=POP_RES)


Salvador, Brazil: WorldPop years available — [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
  2003: 128,057 pixels
  2004: 128,057 pixels
  2005: 128,057 pixels
  2006: 128,057 pixels
  2007: 128,057 pixels
  2008: 128,057 pixels
  2009: 128,057 pixels
  2010: 128,057 pixels
  2011: 128,057 pixels
  2012: 128,057 pixels
  2013: 128,057 pixels
  2014: 128,057 pixels
  2015: 128,057 pixels
  2016: 128,057 pixels
  2017: 128,057 pixels
  2018: 128,057 pixels
  2019: 128,057 pixels
  2020: 128,057 pixels
Saved: data/salvador_population.nc  (18 years, 481x668 pixels)


### Teresina, Brazil

In [10]:
extract_city_population('teresina', DATA_DIR, POP_START, POP_END, resolution=POP_RES)


Teresina, Brazil: WorldPop years available — [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
  2003: 518,377 pixels
  2004: 518,377 pixels
  2005: 518,377 pixels
  2006: 518,377 pixels
  2007: 518,377 pixels
  2008: 518,377 pixels
  2009: 518,377 pixels
  2010: 518,377 pixels
  2011: 518,377 pixels
  2012: 518,377 pixels
  2013: 518,377 pixels
  2014: 518,377 pixels
  2015: 518,377 pixels
  2016: 518,377 pixels
  2017: 518,377 pixels
  2018: 518,377 pixels
  2019: 518,377 pixels
  2020: 518,377 pixels
Saved: data/teresina_population.nc  (18 years, 668x779 pixels)


### Cáceres, Brazil

In [11]:
extract_city_population('caceres', DATA_DIR, POP_START, POP_END, resolution=POP_RES)


Cáceres, Brazil: WorldPop years available — [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
  2003: 790,682 pixels
  2004: 790,682 pixels
  2005: 790,682 pixels
  2006: 790,682 pixels
  2007: 790,682 pixels
  2008: 790,682 pixels
  2009: 790,682 pixels
  2010: 790,682 pixels
  2011: 790,682 pixels
  2012: 790,682 pixels
  2013: 790,682 pixels
  2014: 790,682 pixels
  2015: 790,682 pixels
  2016: 790,682 pixels
  2017: 790,682 pixels
  2018: 790,682 pixels
  2019: 790,682 pixels
  2020: 790,682 pixels
Saved: data/caceres_population.nc  (18 years, 891x891 pixels)


### Verify

In [5]:
from utils.heat import load_population

for city in ['salvador', 'teresina', 'caceres']:
    pop = load_population(city, DATA_DIR)
    total_2020 = float(pop.sel(year=2020).sum())
    print(f'  Total population 2020 : {total_2020:,.0f}')
    print()


salvador population: years 2003–2020, shape {'year': 18, 'lat': 481, 'lon': 668}
  Total population 2020 : 3,169,235

teresina population: years 2003–2020, shape {'year': 18, 'lat': 668, 'lon': 779}
  Total population 2020 : 998,816

caceres population: years 2003–2020, shape {'year': 18, 'lat': 891, 'lon': 891}
  Total population 2020 : 75,301

